# Reasoning Circuits in Language Models: A Mechanistic Interpretation of Syllogistic Inference

**Authors:**  
Geonhee Kim, Marco Valentino, André Freitas

---

This notebook accompanies the paper _"Reasoning Circuits in Language Models: A Mechanistic Interpretation of Syllogistic Inference"_ (Findings of ACL 2025, to appear). It provides code and instructions to reproduce the main results and analyses described in the paper.

For questions or feedback, please contact the authors or visit the [project repository](https://github.com/neuro-symbolic-ai/Mechanistic-Interpretation-Syllogism).

## Table of Contents
1. [Setup](#Setup)
2. [Dataset](#Dataset)
3. [Empirical Evaluation](#Empirical-Evaluation)
4. [Circuit Evaluation](#Circuit-Evaluation)
5. [Transferability](#Transferability)
---

# Setup

In [1]:
# Please clone the repository manually if needed:
# !git clone https://github.com/neuro-symbolic-ai/activation_engineering_syllogistic_reasoning
# %cd ./activation_engineering_syllogistic_reasoning/app

In [2]:
# if you are in colab
%cd sample_data
# path = [github auth token] 
!git clone https://{path}@github.com/neuro-symbolic-ai/Mechanistic-Interpretation-Syllogism
%cd Mechanistic-Interpretation-Syllogism/scripts


[Errno 2] No such file or directory: 'sample_data'
/home/oscb/Mechanistic-Interpretation-Material-Inference/mimi/src/mimi
fatal: destination path 'Mechanistic-Interpretation-Syllogism' already exists and is not an empty directory.


/home/oscb/Mechanistic-Interpretation-Material-Inference/mimi/.direnv/test/lib/python3.9/site-packages/IPython/core/magics/osm.py:393: UserWarning: using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})


/home/oscb/Mechanistic-Interpretation-Material-Inference/mimi/src/mimi/Mechanistic-Interpretation-Syllogism/scripts


/home/oscb/Mechanistic-Interpretation-Material-Inference/mimi/.direnv/test/lib/python3.9/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
# Standard library imports
import os
import sys
import random

# Third-party libraries
import torch as t
from transformer_lens import HookedTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
import importlib

# Project-specific modules (ensure these files exist in ../app/)
from mimi.utils.global_variables import IMAGE_DIR
from mimi.utils.prepare_dataset import MaterialInferenceDataset, Corruption, AMRType
import mimi.utils.helper_functions as h
import mimi.utils.plot_utils as pu
importlib.reload(h)
# Set torch to inference mode by default
t.set_grad_enabled(False)

In [4]:
# Device Setup
if t.cuda.is_available():
    device = t.device("cuda")
    print("Using CUDA GPU:", t.cuda.get_device_name(0))
elif getattr(t.backends, "mps", None) is not None and t.backends.mps.is_available():
    device = t.device("mps")
    print("Using Apple Silicon MPS GPU")
else:
    device = t.device("cpu")
    print("Using CPU")

device

Using CUDA GPU: NVIDIA GeForce RTX 4060 Laptop GPU


device(type='cuda')

In [5]:
# Model Selection and Loading

# Define available models by family
available_models = {
    "gpt2": [
        "gpt2-small", "gpt2-medium", "gpt2-large", "gpt2-xl"
    ],
    "pythia": [
        "pythia-70m", "pythia-160m", "pythia-410m", "pythia-1b", "pythia-6.9b"
    ],
    "qwen": [
        "Qwen/Qwen2.5-0.5B", "Qwen/Qwen2.5-0.5B-Instruct"
    ],
    "llama": [
        "meta-llama/Llama-3.2-1B"
    ]
}

def list_models():
    print("Available models:")
    for family, models in available_models.items():
        print(f"  {family}:")
        for m in models:
            print(f"    - {m}")

def load_model(model_name, device, fold_ln=False):
    print(f"Loading model: {model_name} on device: {device}")
    try:
        model = HookedTransformer.from_pretrained(
            model_name,
            fold_ln=fold_ln,
            device=device
        )
        print("Model loaded successfully.")
        return model
    except Exception as e:
        print(f"Error loading model '{model_name}': {e}")
        return None

# Example usage:
# list_models()  # Uncomment to see all available models

current_model = "gpt2-medium"  # Change this to your desired model
if any(current_model in models for models in available_models.values()):
    model = load_model(current_model, device)
else:
    print(f"Model '{current_model}' is not in the available models list.")
    list_models()

Loading model: gpt2-medium on device: cuda
Loaded pretrained model gpt2-medium into HookedTransformer
Model loaded successfully.


In [6]:
# if fine-tuned model is used
# model_name = "DebateLabKIT/cript-medium"  # Example: 'bert-base-uncased', 'gpt2', etc.
# f_tokenizer = AutoTokenizer.from_pretrained(model_name)
# f_model = AutoModelForCausalLM.from_pretrained(model_name)
# f_model = f_model.to(device)

# model = h.align_fine_tuning_gpt2(model, f_model, device)

# current_model = "cript-medium"

In [7]:
# --- Fine-tuned Model Setup (Optional) ---

fine_tuned_model_name = None #"DebateLabKIT/cript-medium"  # Set to model name string if using a fine-tuned model, else leave as None

def get_base_model_name(fine_tuned_name):
    # Map fine-tuned model names to their base models (extend as needed)
    mapping = {
        "DebateLabKIT/cript-medium": "gpt2-medium",
        # Add more mappings as necessary
    }
    return mapping.get(fine_tuned_name, None)

if fine_tuned_model_name:
    base_model_name = get_base_model_name(fine_tuned_model_name)
    if base_model_name and (current_model == base_model_name):
        print(f"Loading fine-tuned model: {fine_tuned_model_name} (base: {base_model_name})")
        f_tokenizer = AutoTokenizer.from_pretrained(fine_tuned_model_name)
        f_model = AutoModelForCausalLM.from_pretrained(fine_tuned_model_name).to(device)
        # Align the base model with the fine-tuned model
        model = h.align_fine_tuning_gpt2(model, f_model, device)
        current_model = fine_tuned_model_name
    else:
        print(f"[Warning] Fine-tuned model '{fine_tuned_model_name}' is only compatible with base model '{base_model_name}'.")
        print(f"         Please set current_model = '{base_model_name}' before alignment, or update the mapping if using a different fine-tuned model.")
else:
    print("No fine-tuned model specified. Using the base model.")

# --- End Fine-tuned Model Setup ---

No fine-tuned model specified. Using the base model.


In [8]:
# label setup for sequence
image_dir = IMAGE_DIR / current_model
print(f"Image Directory: {image_dir}")

Image Directory: /home/oscb/Mechanistic-Interpretation-Material-Inference/mimi/images/gpt2-medium


# Dataset

In [9]:
import pandas as pd

In [ ]:
s_dataset = MaterialInferenceDataset(N=30, type=AMRType.PRED_SUB)
labels = s_dataset.amr_object.labels

In [11]:
labels

['BEGIN', 'a', '∈_1', 'b1', '∧', 'b2', '∈_2', 'c', '=>', 'a', '->']

In [12]:
#acc = h.evaluate_accuracy(s_dataset.sentences, s_dataset.labels, model, device)
#print(f"Accuracy: {acc:.3f}")


## Localization of Transitive Reasoning Mechanisms

In [13]:
s_prompts = s_dataset.sentences
s_labels = [label[0] for label in s_dataset.labels]
s_second_labels = [label[1] for label in s_dataset.labels]
label_lens, max_len_by_label = s_dataset.amr_object.get_label_token_lengths(s_dataset.prompts)
s_answers = s_dataset.labels
s_pairs = [
    (model.to_tokens(label, prepend_bos=False)[0][0], model.to_tokens(second_label, prepend_bos=False)[0][0]) for label, second_label in s_answers
]
s_answer_tokens = t.stack([t.stack(p) for p in s_pairs])
#h.get_answer_token_sequences(s_labels, s_second_labels, s_dataset.tokenizer, max_len=max_len_by_label["p"]).to(device)

In [14]:
s_answer_tokens

tensor([[48573,  8234],
        [41917, 33408],
        [ 3788,  8300],
        [15050,  8598],
        [ 1964, 11063],
        [ 5366,   275],
        [ 1729,   275],
        [44280, 23642],
        [29824, 15581],
        [30362, 10087],
        [40631,   269],
        [ 6174, 31889],
        [ 1366, 50126],
        [15921, 10214],
        [17243,   275],
        [14492,  8300],
        [40118,  2272],
        [ 3288, 31093],
        [10087,  1196],
        [ 4618, 17797],
        [39930,   458],
        [42560,   649],
        [17659, 27384],
        [35184,  4490],
        [ 3788,  8300],
        [  610, 19435],
        [ 1956,  8598],
        [ 5874,  1242],
        [21622, 24770],
        [ 3788,  5361]], device='cuda:0')

In [ ]:
corrupted_dataset = MaterialInferenceDataset(corruption = Corruption.MID, N=30, type=AMRType.PRED_SUB)

In [16]:
s_pairs = [
    (model.to_tokens(label, prepend_bos=False)[0][0], model.to_tokens(second_label, prepend_bos=False)[0][0]) for label, second_label in corrupted_dataset.labels
]
s_corrupted_answer_tokens = t.stack([t.stack(p) for p in s_pairs])

In [17]:
# Corruption and tokenization
s_corrupted_prompts = corrupted_dataset.sentences
s_tokens = s_dataset.amr_object.get_adjusted_token_sequences(max_len_by_label, s_dataset.prompts).to(device)

s_corrupted_tokens = s_dataset.amr_object.get_adjusted_token_sequences(max_len_by_label, corrupted_dataset.prompts).to(device)

In [18]:
from typing import List, Tuple
import torch as t

def predict_next_words(model, sentence: str, n_words: int = 5) -> List[Tuple[str, float]]:
    """
    Return the top-n_words most likely NEXT TOKENS (as strings) and their logits
    after `sentence` for a transformer_lens HookedTransformer model.
    """
    # Tokenize and move to the model device
    tokens = model.to_tokens(sentence, prepend_bos=False)
    tokens = tokens.to(next(model.parameters()).device)

    # Forward pass to get logits over the vocab at each position
    with t.no_grad():
        logits = model(tokens)  # shape: [batch, seq, d_vocab]

    # Take the logits for the next token (i.e., last position)
    next_token_logits = logits[0, -1, :]  # shape: [d_vocab]

    # Top-k over the vocab
    topk_vals, topk_idx = t.topk(next_token_logits, k=n_words)

    # Decode token ids to readable strings (these are token pieces, not guaranteed whole words)
    token_strs = [model.tokenizer.decode([int(i)]) for i in topk_idx]

    # Return list of (token_string, logit)
    return list(zip(token_strs, [float(v) for v in topk_vals]))



In [19]:
max_len_by_label

{'BEGIN': 2,
 'a': 4,
 '∈_1': 4,
 'b1': 5,
 '∧': 3,
 'b2': 5,
 '∈_2': 4,
 'c': 5,
 '=>': 2,
 '->': 4}

In [20]:
label_lens

[{'BEGIN': 2,
  'a': 2,
  '∈_1': 4,
  'b1': 3,
  '∧': 3,
  'b2': 3,
  '∈_2': 4,
  'c': 2,
  '=>': 2,
  '->': 4},
 {'BEGIN': 2,
  'a': 3,
  '∈_1': 4,
  'b1': 4,
  '∧': 3,
  'b2': 4,
  '∈_2': 4,
  'c': 3,
  '=>': 2,
  '->': 4},
 {'BEGIN': 2,
  'a': 2,
  '∈_1': 4,
  'b1': 3,
  '∧': 3,
  'b2': 3,
  '∈_2': 4,
  'c': 4,
  '=>': 2,
  '->': 4},
 {'BEGIN': 2,
  'a': 3,
  '∈_1': 4,
  'b1': 3,
  '∧': 3,
  'b2': 3,
  '∈_2': 4,
  'c': 4,
  '=>': 2,
  '->': 4},
 {'BEGIN': 2,
  'a': 4,
  '∈_1': 4,
  'b1': 5,
  '∧': 3,
  'b2': 5,
  '∈_2': 4,
  'c': 3,
  '=>': 2,
  '->': 4},
 {'BEGIN': 2,
  'a': 3,
  '∈_1': 4,
  'b1': 3,
  '∧': 3,
  'b2': 3,
  '∈_2': 4,
  'c': 4,
  '=>': 2,
  '->': 4},
 {'BEGIN': 2,
  'a': 2,
  '∈_1': 4,
  'b1': 5,
  '∧': 3,
  'b2': 5,
  '∈_2': 4,
  'c': 5,
  '=>': 2,
  '->': 4},
 {'BEGIN': 2,
  'a': 4,
  '∈_1': 4,
  'b1': 3,
  '∧': 3,
  'b2': 3,
  '∈_2': 4,
  'c': 3,
  '=>': 2,
  '->': 4},
 {'BEGIN': 2,
  'a': 2,
  '∈_1': 4,
  'b1': 3,
  '∧': 3,
  'b2': 3,
  '∈_2': 4,
  'c': 3,
  '=>'

In [21]:
list(s_dataset.amr_object.B_col)

['birds',
 'fish',
 'trees',
 'flowering plants',
 'programming languages',
 'cities',
 'monuments',
 'metals',
 'monosaccharides',
 'crystals',
 'noble gases',
 'big cats',
 'avians',
 'marsupials',
 'constrictor snakes',
 'insects',
 'cephalopods',
 'cetaceans',
 'cnidarians',
 'fungi',
 'bacteria',
 'conifers',
 'bryophytes',
 'oilseed crops',
 'cereal grains',
 'rice varieties',
 'cheeses',
 'fermented foods',
 'breads',
 'coffee drinks',
 'sausage dishes',
 'soy foods',
 'seafood dishes',
 'botanical berries',
 'culinary vegetables',
 'tree nuts',
 'pseudocereals',
 'sweeteners',
 'deserts',
 'brazil',
 'mountain ranges',
 'state capitals',
 'grand-duchy',
 'italian cities',
 'european capitals',
 'mountain range',
 'deserts of asia',
 'oceans',
 'mountains',
 'dog breeds',
 'cat breeds',
 'gemstones',
 'gem-quality minerals',
 'sedimentary rocks',
 'igneous rocks',
 'plutonic rocks',
 'carbonate rocks',
 'nerve cells',
 'liver cells',
 'red blood cells',
 'peptide hormones',
 'pr

In [22]:
predict_next_words(model, "because fork  refracts light and because prism is a kind of optical element thus prism is a kind of optical element that", 10)

[(' ref', 16.64028549194336),
 (' can', 15.960257530212402),
 (' is', 15.81367301940918),
 (' has', 15.17459774017334),
 (' reflects', 14.929160118103027),
 (' diff', 14.259773254394531),
 (' bends', 14.171603202819824),
 (' does', 13.959643363952637),
 (' mirrors', 13.627065658569336),
 (' allows', 13.621684074401855)]

In [23]:
s_prompts[1]

'because  sandstone belongs to   sedimentary rocks and because  sedimentary rocks belongs to   geological materials thus  sandstone belongs to  '

In [24]:
s_corrupted_prompts[1]

'because  sandstone belongs to   sedimentary rocks and because red belongs to   geological materials thus  sandstone belongs to  '

In [25]:
corrupted_dataset.labels[16]

(' observational instruments', ' space telescopes')

In [26]:
s_dataset.tokenizer.encode("couriers", add_special_tokens=False)

[66, 280, 8910]

In [27]:
space_id = s_dataset.tokenizer.encode(" ", add_special_tokens=False)[0]

In [28]:
#s_clean_logits,  s_clean_cache  = h.run_ignoring_pad(model, s_tokens, s_dataset.tokenizer.pad_token_id)
#s_corrupted_logits, s_corrupted_cache = h.run_ignoring_pad(model, s_corrupted_tokens, s_dataset.tokenizer.pad_token_id)

In [29]:
# Compute logits for interventions
# Compute logits for interventions
s_clean_logits,  s_clean_cache  = model.run_with_cache(s_tokens)
s_corrupted_logits, s_corrupted_cache = model.run_with_cache(s_corrupted_tokens)

s_clean_logprob = h.batch_avg_logprob_for_labels(model, s_dataset.tokenizer, s_prompts,            s_labels, device)
s_corr_logprob  = h.batch_avg_logprob_for_labels(model, s_dataset.tokenizer, s_corrupted_prompts, s_labels, device)

# Ensure they're plain floats (in case your helper returns a 0-D tensor)

print(f"Clean avg log-prob:     {s_clean_logprob:.4f}")
print(f"Corrupted avg log-prob: {s_corr_logprob:.4f}")



Clean avg log-prob:     -0.9672
Corrupted avg log-prob: -1.4994


In [30]:
last = s_clean_logits[:, -1, :]
answer_logits = last.gather(dim=-1, index=s_answer_tokens)
correct_logits, incorrect_logits = answer_logits.unbind(dim=-1)
answer_logit_diff = correct_logits - incorrect_logits

In [31]:
c_last = s_corrupted_logits[:, -1, :]
c_answer_logits = c_last.gather(dim=-1, index=s_answer_tokens)
c_correct_logits, c_incorrect_logits = c_answer_logits.unbind(dim=-1)
c_answer_logit_diff = c_correct_logits - c_incorrect_logits

In [32]:
answer_logit_diff

tensor([ 0.5042, -0.7118,  0.2252, -2.8182, -0.6005,  1.7014,  1.8766, -2.4390,
         2.4670, -1.0058, -0.7709,  0.6656,  0.8973,  2.4413, -0.8105, -0.8080,
        -0.5210,  1.3519,  2.1672,  2.4028, -1.3978,  2.3482,  2.0806,  1.0924,
        -0.2276, -0.3827,  1.8207,  1.4250,  1.6106,  0.9749], device='cuda:0')

In [33]:
c_answer_logit_diff

tensor([ 0.2440, -0.4625, -0.6094, -3.3031, -1.9909,  0.6686,  1.7720, -2.0739,
         2.3578, -0.9609,  0.1252,  0.2168,  0.4500,  2.4302, -0.0700, -1.7388,
        -0.5526,  0.8831,  2.5054,  2.7384, -0.5157,  2.6555,  1.1142,  1.7077,
        -0.7830, -0.0518,  1.8542,  0.8698,  1.2736,  1.0299], device='cuda:0')

In [34]:
clean_lead = answer_logit_diff - c_answer_logit_diff
clean_lead

tensor([ 0.2602, -0.2493,  0.8347,  0.4850,  1.3904,  1.0328,  0.1046, -0.3650,
         0.1092, -0.0449, -0.8961,  0.4487,  0.4473,  0.0110, -0.7405,  0.9308,
         0.0316,  0.4688, -0.3382, -0.3356, -0.8821, -0.3073,  0.9663, -0.6154,
         0.5553, -0.3308, -0.0335,  0.5552,  0.3370, -0.0551], device='cuda:0')

In [35]:
# keep items where clean_lead > 0
idx = (clean_lead < 0).nonzero(as_tuple=True)[0].tolist()
filtered_prompts = [s_prompts[i] for i in idx]
filtered_prompts

['because  sandstone belongs to   sedimentary rocks and because  sedimentary rocks belongs to   geological materials thus  sandstone belongs to  ',
 'because  andromeda galaxy belongs to   spiral galaxies and because  spiral galaxies belongs to   galactic systems thus  andromeda galaxy belongs to  ',
 'because  sparrow belongs to   birds and because  birds belongs to   vertebrates thus  sparrow belongs to  ',
 'because  coral belongs to   cnidarians and because  cnidarians belongs to   invertebrates thus  coral belongs to  ',
 'because  blue whale belongs to   baleen whales and because  baleen whales belongs to   megafauna thus  blue whale belongs to  ',
 'because  penguin belongs to   avians and because  avians belongs to   birds thus  penguin belongs to  ',
 'because  tofu belongs to   soy foods and because  soy foods belongs to   plant-based proteins thus  tofu belongs to  ',
 'because  granite belongs to   plutonic rocks and because  plutonic rocks belongs to   intrusive igneous ro

In [36]:
idx

[1, 7, 9, 10, 14, 18, 19, 20, 21, 23, 25, 26, 29]

In [37]:
pred_ids = t.argmax(last, dim=-1)
s_dataset.tokenizer.decode(pred_ids)


' flowering  software mountain euro south non spiral inert birds  meat data fruits b programming the natural birds plant pl endemic dairy monarchy programming insects land visual minerals software'

In [38]:
pred_ids

tensor([48573,   220,  3788,  8598, 11063,  5366,  1729, 23642, 29824, 10087,
          220,  6174,  1366, 15921,   275,  8300,   262,  3288, 10087,  4618,
          458, 42560, 17659, 35184,  8300, 19435,  1956,  5874, 21622,  3788],
       device='cuda:0')

In [39]:
s_tokens

tensor([[13893,   220, 19450,  ..., 28079,   284,   220],
        [13893,   220,  6450,  ..., 28079,   284,   220],
        [13893,   220, 17000,  ..., 28079,   284,   220],
        ...,
        [13893,   220,  3056,  ..., 28079,   284,   220],
        [13893,   220, 47969,  ..., 28079,   284,   220],
        [13893,   220, 32639,  ..., 28079,   284,   220]], device='cuda:0')

In [40]:
s_dataset.tokenizer.decode(s_tokens[13])

'because  avocado   belongs to  botanical berries  and because  botanical berries  belongs to  fruits    thus  avocado   belongs to '

In [41]:
#s_answer_tokens

In [42]:
s_answers

[(' flowering', ' fruit tree'),
 (' geological materials', ' sedimentary rocks'),
 (' software engineering tools', ' programming languages'),
 (' cordilleras', ' mountain range'),
 (' political centers', ' european capitals'),
 (' south america', ' brazil'),
 (' nonvascular plants', ' bryophytes'),
 (' galactic systems', ' spiral galaxies'),
 (' inert elements', ' noble gases'),
 (' vertebrates', ' birds'),
 (' invertebrates', ' cnidarians'),
 (' meat products', ' sausage dishes'),
 (' data management systems', ' relational databases'),
 (' fruits', ' botanical berries'),
 (' megafauna', ' baleen whales'),
 (' computing technologies', ' programming languages'),
 (' observational instruments', ' space telescopes'),
 (' natural satellites', ' moons of jupiter'),
 (' birds', ' avians'),
 (' plant-based proteins', ' soy foods'),
 (' intrusive igneous rocks', ' plutonic rocks'),
 (' endemic fauna', ' new zealand birds'),
 (' dairy products', ' cheeses'),
 (' monarchy', ' grand-duchy'),
 (' 

In [43]:
#s_dataset.tokenizer.decode([7353,  66])

In [44]:
s_clean_logit_diff     = h.compute_logit_diff(s_clean_logits,     s_answer_tokens)
s_corrupted_logit_diff = h.compute_logit_diff(s_corrupted_logits, s_answer_tokens)

print(f"Clean logit diff: {s_clean_logit_diff:.4f}")
print(f"Corrupted logit diff: {s_corrupted_logit_diff:.4f}")

Clean logit diff: 0.5186
Corrupted logit diff: 0.3928


### Figure 4(a) attention head output patching

In [45]:
del s_corrupted_cache, s_corrupted_logits   # if not needed later
t.cuda.empty_cache()

In [46]:
s_attn_denoising = h.patching_attention(model, s_corrupted_tokens, s_clean_cache, h.metric_denoising, s_answer_tokens, s_clean_logit_diff, s_corrupted_logit_diff, 'z', device)
norm_s_attn_denoising = h.normalise_tensor(s_attn_denoising)

100%|██████████| 24/24 [01:05<00:00,  2.74s/it]


In [47]:
sequence = h.build_head_sequence_from_map(norm_s_attn_denoising)
abs_sequence = h.build_head_sequence_from_map(norm_s_attn_denoising, absolute = True)


In [48]:
sequence

[(11, 10),
 (12, 3),
 (7, 2),
 (19, 1),
 (17, 12),
 (10, 0),
 (9, 3),
 (13, 10),
 (5, 8),
 (8, 1),
 (10, 14),
 (6, 15),
 (12, 8),
 (8, 12),
 (6, 1)]

In [49]:
pu.plot_attn(norm_s_attn_denoising, labels, image_dir / s_dataset.type.value / "symbolic_denoising.png")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Plot saved to /home/oscb/Mechanistic-Interpretation-Material-Inference/mimi/images/gpt2-medium/argument_substitution/symbolic_denoising.png


In [50]:
del s_attn_denoising, norm_s_attn_denoising

### Figure 4(b) residual stream patching

In [51]:
s_resid_denoising = h.patching_residual(model, s_corrupted_tokens, s_clean_cache, h.metric_denoising, s_answer_tokens, s_clean_logit_diff, s_corrupted_logit_diff, device)





100%|██████████| 24/24 [03:12<00:00,  8.00s/it]


In [52]:

# 2) build cumulative spans (sequence axis)
spans = h.build_label_spans(max_len_by_label, labels)

# 3) compress your residual heatmap
#    s_resid_denoising2 is typically [n_layers, seq_len]; function handles both orders
resid_to_plot = h.compress_by_label_spans(s_resid_denoising.T, spans, labels, mode="mean")

# 4) normalize (your helper) and plot (your existing code stays unchanged)
resid_norm = h.normalise_tensor(resid_to_plot)  # torch tensor OK
pu.plot_residual(resid_norm.T, labels, image_dir / s_dataset.type.value / "symbolic_residual.png")


Plot saved to /home/oscb/Mechanistic-Interpretation-Material-Inference/mimi/images/gpt2-medium/argument_substitution/symbolic_residual.png


In [53]:
del s_resid_denoising

### Figure 4(c) OV logit lens

In [54]:
element_list = s_dataset.amr_object.element_list

In [55]:
token_alpha = model.to_tokens(element_list, prepend_bos=False).to(device)
token_alpha = [ element[0].item() for element in token_alpha ]

In [56]:
deductive_head_ov = model.OV
ev = model.W_E
uev = model.W_U
ov_circuit = ev.cpu()[token_alpha, : ] @ deductive_head_ov.AB.cpu() @ uev.cpu()[:, token_alpha]

In [57]:
# OV circuit for head 11.10
layer = sequence[0][0]
head = sequence[0][1]

pu.plot_attn(h.normalise_tensor(ov_circuit[layer, head, : , :]), element_list, image_dir / s_dataset.type.value / "symbolic_ov_circuit.png")

Plot saved to /home/oscb/Mechanistic-Interpretation-Material-Inference/mimi/images/gpt2-medium/argument_substitution/symbolic_ov_circuit.png


## Localization of Term-Related Information Flow

In [ ]:
all_corrupted_dataset = MaterialInferenceDataset(corruption = Corruption.ALL, N=30, type=AMRType.PRED_SUB)

In [59]:
# Corruption and tokenization
s_all_corrupted_prompts = all_corrupted_dataset.sentences
s_all_corrupted_tokens = all_corrupted_dataset.amr_object.get_adjusted_token_sequences(max_len_by_label, all_corrupted_dataset.prompts).to(device)

In [60]:
# Compute logits for interventions
s_all_corrupted_logits, s_all_corrupted_cache = model.run_with_cache(s_all_corrupted_tokens)

s_clean_logit_diff = h.compute_logit_diff(s_clean_logits, s_answer_tokens)
print(f"Clean logit diff: {s_clean_logit_diff:.4f}")

s_all_corrupted_logit_diff = h.compute_logit_diff(s_all_corrupted_logits, s_answer_tokens)
print(f"Corrupted logit diff: {s_all_corrupted_logit_diff:.4f}")

Clean logit diff: 0.5186
Corrupted logit diff: -1.0946


### Figure 4(d) residual stream patching

In [61]:
s_tokens.shape


torch.Size([30, 42])

In [62]:
s_all_corrupted_tokens.shape

torch.Size([30, 42])

In [63]:
# keep s_clean_cache; drop the rest before the second patching

del s_all_corrupted_cache, s_all_corrupted_logits   # if not needed later
t.cuda.empty_cache()


In [64]:
s_resid_denoising2 = h.patching_residual(model, s_all_corrupted_tokens, s_clean_cache, h.metric_denoising, s_answer_tokens, s_clean_logit_diff, s_all_corrupted_logit_diff, device)


100%|██████████| 24/24 [03:19<00:00,  8.30s/it]


In [65]:
resid_to_plot2 = h.compress_by_label_spans(s_resid_denoising2.T, spans, labels, mode="mean")

# 4) normalize (your helper) and plot (your existing code stays unchanged)
resid_norm2 = h.normalise_tensor(resid_to_plot2)  # torch tensor OK
pu.plot_residual(resid_norm2.T, labels, image_dir / s_dataset.type.value / "symbolic_residual2.png")

Plot saved to /home/oscb/Mechanistic-Interpretation-Material-Inference/mimi/images/gpt2-medium/argument_substitution/symbolic_residual2.png


In [66]:
del s_resid_denoising2

# Circuit Evaluation

### Figure 5(a) correctness of the circuit

In [67]:
# necessity of symbolic
necessity_score = h.necessity_check(model, s_labels, s_tokens, s_answer_tokens, s_clean_logit_diff, 'mean', device)
sufficiency_score = h.sufficiency_check(model, s_labels, s_tokens, s_answer_tokens, s_clean_logit_diff, 'mean', device)

100%|██████████| 11/11 [00:02<00:00,  4.81it/s]


In [68]:
pu.plot_ablation(s_clean_logit_diff, necessity_score, sufficiency_score, image_dir / s_dataset.type.value / "correctness_of_circuit.png")

Plot saved to /home/oscb/Mechanistic-Interpretation-Material-Inference/mimi/images/gpt2-medium/argument_substitution/correctness_of_circuit.png


In [69]:

custom_necessity_score = h.necessity_check(model, s_labels, s_tokens, s_answer_tokens, s_clean_logit_diff, 'mean', device, sequence=sequence)
custom_sufficiency_score = h.sufficiency_check(model, s_labels, s_tokens, s_answer_tokens, s_clean_logit_diff, 'mean', device, sequence=sequence)

100%|██████████| 15/15 [00:03<00:00,  4.89it/s]


In [70]:
pu.plot_ablation(s_clean_logit_diff, custom_necessity_score, custom_sufficiency_score, image_dir / s_dataset.type.value / "correctness_of_circuit_top15.png")

Plot saved to /home/oscb/Mechanistic-Interpretation-Material-Inference/mimi/images/gpt2-medium/argument_substitution/correctness_of_circuit_top15.png


In [71]:
absolute_necessity_score = h.necessity_check(model, s_labels, s_tokens, s_answer_tokens, s_clean_logit_diff, 'mean', device, sequence=abs_sequence)
absolute_sufficiency_score = h.sufficiency_check(model, s_labels, s_tokens, s_answer_tokens, s_clean_logit_diff, 'mean', device, sequence=abs_sequence)

100%|██████████| 15/15 [00:03<00:00,  4.71it/s]


In [72]:
pu.plot_ablation(s_clean_logit_diff, absolute_necessity_score, absolute_sufficiency_score, image_dir / s_dataset.type.value / "correctness_of_circuit_top15abs.png")

Plot saved to /home/oscb/Mechanistic-Interpretation-Material-Inference/mimi/images/gpt2-medium/argument_substitution/correctness_of_circuit_top15abs.png


### Figure 5(b) robustness of the circuit

In [73]:
# numeric perturbed dataset
n_dataset = SyllogismDataset(
            N=N/6, # because of permutation
            seed=seed,
            device=device,
            type='numeric',
            template_type='AAA1'
        )
for s, l in zip(n_dataset.sentences[:6], n_dataset.labels[:6]):
    print(f'{s} => {l}')


NameError: name 'SyllogismDataset' is not defined

In [ ]:
# Corruption (option)
def perturb_quantifier(prompts, As, Bs, labels):
    candidates = [' Every', ' Every', ' Each', ' Each', ' All']
    corrupted_prompts = []
    for i in range(len(prompts)):
        prompt = prompts[i]
        label = labels[i]
        corrupted_labels = []
        A = As[i]
        B = Bs[i]
        new_list = list(filter(lambda x: x not in [A, B, label], candidates))
        target = random.sample(new_list, 2)
        a_be = ' are' if target[0] == ' All' else ' is'
        b_be = ' are' if target[1] == ' All' else ' is'
        corrupted_prompt = prompt.replace(' All' + B + ' are' , target[0] + B + a_be).replace('All' + A + ' are', target[1][1:] + A + b_be)
        corrupted_prompts.append(corrupted_prompt)
        corrupted_labels.append(target)

    return corrupted_prompts, corrupted_labels


: 

: 

: 

In [ ]:
n_prompts = n_dataset.sentences
n_labels = n_dataset.labels
n_second_labels = n_dataset.B

n_answers = list(zip(n_labels, n_second_labels))
n_answer_tokens = t.concat([
    model.to_tokens(names, prepend_bos=False).T for names in n_answers
])

: 

: 

: 

In [ ]:
q_prompts = s_dataset.sentences
q_labels = s_dataset.labels
q_As = s_dataset.A
q_Bs = s_dataset.B

q_corrupted_prompts, _ = perturb_quantifier(q_prompts, q_As, q_Bs, q_labels)

q_answers = list(zip(q_labels, q_Bs))
q_answer_tokens = t.concat([
    model.to_tokens(names, prepend_bos=False).T for names in q_answers
])

: 

: 

: 

In [ ]:
# tokenisation
n_tokens = model.to_tokens(n_prompts, prepend_bos=False).to(device)
q_tokens = model.to_tokens(q_prompts, prepend_bos=False).to(device)

: 

: 

: 

In [ ]:
# Compute logits for interventions
n_clean_logits, n_clean_cache = model.run_with_cache(n_tokens)
q_clean_logits, q_clean_cache = model.run_with_cache(n_tokens)

n_clean_logit_diff = h.compute_logit_diff(n_clean_logits, n_answer_tokens)
print(f"Clean logit diff: {n_clean_logit_diff:.4f}")

q_clean_logit_diff = h.compute_logit_diff(q_clean_logits, q_answer_tokens)
print(f"Clean logit diff: {q_clean_logit_diff:.4f}")

: 

: 

: 

In [ ]:
# necessity, sufficiency of numeric perturbation
n_necessity_score = h.necessity_check(model, n_labels, n_tokens, n_answer_tokens, n_clean_logit_diff, 'mean', device)
n_sufficiency_score = h.sufficiency_check(model, n_labels, n_tokens, n_answer_tokens, n_clean_logit_diff, 'mean', device)

: 

: 

: 

In [ ]:
# necessity, sufficiency of quantifier perturbation
q_necessity_score = h.necessity_check(model, q_labels, q_tokens, q_answer_tokens, q_clean_logit_diff, 'mean', device)
q_sufficiency_score = h.sufficiency_check(model, q_labels, q_tokens, q_answer_tokens, q_clean_logit_diff, 'mean', device)

: 

: 

: 

In [ ]:
pu.plot_ablation_robust(y1=n_necessity_score, y2=n_sufficiency_score, y3=q_necessity_score, y4 = q_sufficiency_score,  baseline1= n_clean_logit_diff, baseline2=q_clean_logit_diff, save_path=image_dir / s_dataset.type.value / "robustness_of_circuit.png", title="")


: 

: 

: 

# Circuit Transferability

In [ ]:
# setup
bc_prompts = bc_dataset.sentences
bc_labels = bc_dataset.labels
bc_second_labels = bc_dataset.B

bc_answers = list(zip(bc_labels, bc_second_labels))
bc_answer_tokens = t.concat([
    model.to_tokens(names, prepend_bos=False).T for names in bc_answers
])

bi_prompts = bi_dataset.sentences
bi_labels = bi_dataset.labels
bi_second_labels = bi_dataset.B

bi_answers = list(zip(bi_labels, bi_second_labels))
bi_answer_tokens = t.concat([
    model.to_tokens(names, prepend_bos=False).T for names in bi_answers
])

# tokenisation
bc_tokens = model.to_tokens(bc_prompts, prepend_bos=False).to(device)
bi_tokens = model.to_tokens(bi_prompts, prepend_bos=False).to(device)

: 

: 

: 

In [ ]:
# Compute logits for interventions
bc_clean_logits, bc_clean_cache = model.run_with_cache(bc_tokens)
bi_clean_logits, bi_clean_cache = model.run_with_cache(bi_tokens)

bc_clean_logit_diff = h.compute_logit_diff(bc_clean_logits, bc_answer_tokens)
print(f"Clean logit diff: {bc_clean_logit_diff:.4f}")

bi_clean_logit_diff = h.compute_logit_diff(bi_clean_logits, bi_answer_tokens)
print(f"Clean logit diff: {bi_clean_logit_diff:.4f}")

: 

: 

: 

In [ ]:
# belief-consistent
bc_necessity_score = h.necessity_check(model, bc_labels, bc_tokens, bc_answer_tokens, bc_clean_logit_diff, 'mean', device)
bc_sufficiency_score = h.sufficiency_check(model, bc_labels, bc_tokens, bc_answer_tokens, bc_clean_logit_diff, 'mean', device)

: 

: 

: 

In [ ]:
# belief-inconsistent
bi_necessity_score = h.necessity_check(model, bi_labels, bi_tokens, bi_answer_tokens, bi_clean_logit_diff, 'mean', device)
bi_sufficiency_score = h.sufficiency_check(model, bi_labels, bi_tokens, bi_answer_tokens, bi_clean_logit_diff, 'mean', device)

: 

: 

: 

### Figure 6(a) belief-consistent

In [ ]:
pu.plot_ablation(bc_clean_logit_diff, bc_necessity_score, bc_sufficiency_score, image_dir / "correctness_of_circuit_bc.png")

: 

: 

: 

### Figure 6(b) belief-inconsistent

In [ ]:
pu.plot_ablation(bi_clean_logit_diff, bi_necessity_score, bi_sufficiency_score, image_dir / "correctness_of_circuit_bi.png")

: 

: 

: 

### Table 2 all unconditionally valid syllogisms

In [ ]:
# ordered by accuracy
moods_ordered = ['AII3', 'IAI3', 'IAI4', 'AAA1', 'EAE1', 'EIO4', 'EIO3', 'AII1', 'AOO2', 'AEE4', 'OAO3', 'EIO1', 'EIO2', 'EAE2', 'AEE2']

: 

: 

: 

In [ ]:
# all symbolic datasets import
s_datasets = {}

for template in moods_ordered:
    s_dataset = SyllogismDataset(
            N=N,
            seed=seed,
            device=device,
            type='symbolic',
            template_type=template
        )
    s_datasets[template] = s_dataset


: 

: 

: 

In [ ]:
# setup
mood_dics = []

for template in moods_ordered:
    mood_dic = {
        'mood': template,
        'prompts': s_datasets[template].sentences,
        'labels': s_datasets[template].labels,
        'second_labels': s_datasets[template].B

    }
    s_answers = list(zip(mood_dic['labels'], mood_dic['second_labels']))
    mood_dic['answers'] = s_answers
    s_answer_tokens = t.concat([
        model.to_tokens(names, prepend_bos=False).T for names in s_answers
    ])
    mood_dic['answer_tokens'] = s_answer_tokens

    mood_dics.append(mood_dic)

: 

: 

: 

In [ ]:
# get base logit diff for 15 syllogisms
for i, template in enumerate(moods_ordered):
    tokens = model.to_tokens(mood_dics[i]['prompts'], prepend_bos=False).to(device)
    clean_logit_diff = h.get_batched_logit_diff(5, tokens, mood_dics[i]['answer_tokens'], model)
    mood_dics[i]['tokens'] = tokens
    mood_dics[i]['clean_logit_diff'] = clean_logit_diff

    bia_diff = 0
    for correct, wrong in mood_dics[i]['answer_tokens']:
        diff = model.unembed.b_U[correct.item()] - model.unembed.b_U[wrong.item()]
        bia_diff += diff

    bia_diff = bia_diff/len(mood_dics[i]['prompts'])
    mood_dics[i]['bia_diff'] = bia_diff
    print(mood_dics[i]['clean_logit_diff'], mood_dics[i]['bia_diff'])

: 

: 

: 

In [ ]:
# ablation
for i, template in enumerate(moods_ordered):
    mean_scores = h.necessity_check(model, mood_dics[i]['labels'], mood_dics[i]['tokens'], mood_dics[i]['answer_tokens'], mood_dics[i]['clean_logit_diff'], 'mean', device)
    sf_mean_scores = h.sufficiency_check(model, mood_dics[i]['labels'], mood_dics[i]['tokens'], mood_dics[i]['answer_tokens'], mood_dics[i]['clean_logit_diff'], 'mean',device)
    mood_dics[i]['mean_scores'] = mean_scores
    mood_dics[i]['sf_mean_score'] = sf_mean_scores

: 

: 

: 

In [ ]:
pu.plot_ablation_syllogisms(mood_dics = mood_dics, save_path=image_dir / "syllogism_ablation.png", title="",)

: 

: 

: 